# Why does the trained decomposition of a broad fine-tune fail the adversarial check, and do SGD or LoRA updates do better?

*E39, `experiments/broad_fragility`, 2026-10-27, with Amendment 1, the SGD arm at smaller rates. Record:
`reports/analyses/2026-10-27_analysis_broad-fragility.md`. Plan: `reports/plans/2026-10-27_plan_broad-fragility.md`.
Follows E37 and its Amendment 1, `reports/analyses/2026-10-26_analysis_teacher-spread.md`. Independently checked
2026-10-27, `reports/reviews/2026-10-27_review_broad-fragility-analysis.md`; corrections applied. Amendment 1 was
added after the check. Figures from `experiments/broad_fragility/plot.py`.*

**Words.** **The decomposition after teacher training**, "the trained decomposition" for short, is the released
decomposition with its alive subcomponents trained to imitate a fine-tuned model. Γ and Δ stay frozen. **The check**
is the paper's adversarial check: a 20-step mask adversary from one random start on 256 Pile rows. It is read as the
state's KL over the unchanged decomposition's KL, so 1 means no more fragile than the original. It passes at or under
1.40 and fails at or over 1.61. **The excess** is how far the check sits above 1.

## The answer

The user asked whether the same holds for SGD or LoRA updates, which are less spread out. Not in general. Only the
LoRA's trained decomposition passes the check, at 1.27. SGD's update is nearly as concentrated, with a stable rank of
3.5 against the LoRA's 1.9 and Adam's 28.5, and changes ordinary text the least. Yet its trained decomposition fails
at 2.07, like Adam's 2.20. So stable rank does not decide it, and what sets the LoRA apart is untested. On the Adam
update, the excess needs most of the MLP changes together. Isotropic noise of the same weight size passed, but it
changed the outputs a twenty-fifth as much.

> **Something unexpected happened that I cannot explain.** Three filed predictions went the other way. A random move
> of the trained decomposition's size passed. The rarely important subcomponents mattered least. The LoRA's trained
> decomposition passed. A fourth, that no single layer carries most of the excess, missed narrowly. One result had no
> prediction: the MLP changes alone carried all the excess, and the attention changes hid part of it. Each is set out
> under Surprises.

## What we were trying to test

E37 trained the subcomponents to imitate a fine-tune that teaches the 67M model to write lowercase text, a change at
every position. The trained decomposition kept the whole gain but read 2.20 under the check. Training four times
longer did not bring it down. The user asked for cheap tests of why, and: "does the same apply for SGD or LoRA where
the updates are less diffuse?" All tests start from E37's trained decomposition, retrained inside the job.

- **Random moves.** Each matrix's alive $U$ and $V$ get Gaussian noise, scaled so each matrix's weight change has the
  size of the trained decomposition's change there. Two noise seeds at full size, one at half size. If the check
  punishes any move of this size, these fail too, and the failure is brittleness.
- **Reverts.** Put the released $U$ and $V$ back on a group of subcomponents and keep the trained values on the rest.
  The groups are each layer, all attention matrices, and all MLP matrices. Then, per matrix, halves of the alive
  subcomponents ranked by Γ's mean causal importance on the check's rows. The **rarely important half** tests one
  guess. Teacher training may have written part of the change into subcomponents Γ hardly ever switches on, which
  the adversary can then switch off cheaply. The **often important half** is the rest, and a **random half** is the
  null. A revert is scored as the **share of the excess removed**, $(r_\text{trained} - r_\text{revert})/(r_\text{trained} - 1)$
  with $r$ the check. 1 removes it all, 0 none, and a negative share makes it worse.
- **Other update shapes.** The same fine-tune made two more ways, both on all 24 matrices: plain SGD, and a rank-8
  LoRA trained with Adam. Each takes the grid rate whose held-out gain is nearest the Adam fine-tune's. The first SGD
  grid was too large, so Amendment 1 reran SGD on a smaller one. Each update is placed three ways: **do-nothing**,
  which leaves it in Δ; the **scaled adjoint split**; and **teacher training** by E37's recipe. **Stable rank**,
  $\|\delta W\|_F^2/\|\delta W\|_2^2$ averaged over matrices, says how many directions an update spreads over.

Two more numbers are read at every state. **Behaviour kept** is the share of the fine-tune's held-out gain in mean
log-probability that the state keeps: 0 is the unchanged model, 1 the fine-tune. **Damage** is
$\mathrm{KL}(W_0 \,\|\, \text{state})$ on plain Pile text, in nats per token.

**Controls, run first, with an early stop.** All passed in both jobs. The unchanged model gave the card's known seed
spread of 0.0748 and E37's readings to four digits. The retrained decomposition read 2.2011, E37's value, and kept
0.998 of the behaviour. Putting every subcomponent back read exactly as the unchanged model.

## What we expected, and why

- **A random move of the same size fails**, at 1.8 to 3. E37's fixes all ended near 2 whatever the route, which looked
  like a property of moving this far.
- **The half-size move reads lower**, a sanity check on the move.
- **The excess is spread over layers**, so no layer's revert removes more than 0.6 of it.
- **The rarely important half removes at least 0.2 more than the random half**, if the frozen Γ is the weak point.
- **The SGD and LoRA updates are less spread than Adam's by stable rank.** This is the third test's premise.
- **Their trained decompositions still fail.** If the failure belongs to broad behaviour, update shape should not
  matter.
- **Their do-nothing states fail**, since the whole update sits in Δ, which the adversary can switch off.
- **Amendment 1: the chosen SGD rate's gain is within 30% of Adam's**, so the SGD arm is comparable at all.

## Does a random move of the same size fail the check?

![random moves](../experiments/broad_fragility/figures/moves.png)

**Reading it.** No. Both full-size moves read 1.03 and 1.07, inside the grey band where the unchanged model sits
across adversary seeds. The half-size move reads 0.99. The trained decomposition, moved by the same amount in every
matrix, reads 2.20. So the check does not punish an isotropic move of this weight size. The right panel limits what
that says. The random moves barely change what the model predicts on plain text: their damage is about a
twenty-fifth of the trained decomposition's. A pass is what both readings predict: "the update is the problem" and
"the decomposition is fragile along the directions training moves it in". In E37's last amendment, a masked recipe
with nothing to imitate moved the subcomponents half as far and failed at 3.14. Only a random move matched in output
change would separate the two readings.

## Which subcomponents carry the excess?

![reverts](../experiments/broad_fragility/figures/reverts.png)

**Reading it.** The MLP subcomponents' changes carry it. Putting back every MLP matrix removes all of the excess,
reading 1.00. Putting back every attention matrix instead raises the check from 2.20 to 2.70, so the attention changes
were offsetting part of it. By layer, layer 0 removes 0.70 of the excess and layer 3 only 0.11; the four shares add to
1.44, so the effects do not add up. Reverting a random half of each matrix, or the often important half, removes
nearly all of it, 0.97 and 0.96. The rarely important half removes only 0.63, the opposite of the guess it tested.
The halves span all 24 matrices; halves within the MLP alone were not run.

The right panel shows the price in behaviour. Putting back the MLP matrices keeps less than half the behaviour. The
random half keeps all of it by mean log-probability, and passes. That is not a fix yet. Its KL to the fine-tuned model
rises from 0.012 to 0.075 nats, where the unchanged model reads 0.272. Two caveats hold for the whole figure. Γ's
importance was read at the unchanged model, and even the often important half averages only 0.028, so the ranking may
not separate what the adversary can switch off. And every revert that cut the damage on plain text to 0.054 or less
passed, so "the MLP carries it" and "the check falls with the remaining damage" both fit.

## Does a less spread-out update give a less fragile trained decomposition?

![update shape](../experiments/broad_fragility/figures/shape.png)

**Reading it.** For the LoRA, yes; for SGD, no. Teacher training on the LoRA reads 1.27, under the pass line, keeping
0.96 of its behaviour. On the Adam fine-tune it reads 2.20, and on the SGD fine-tune 2.07. Do-nothing fails for all
three, and the adjoint split passes for all three while keeping less behaviour. The SGD and LoRA fine-tunes are a
close pair: their do-nothing checks are 2.62 and 2.65, and their gains 0.137 and 0.148. Yet after the same teacher
training one fails and one passes.

The first run's SGD grid, 0.3 to 3, was too large: every rate lowered held-out log-probability, so the rest of that
arm was skipped. Amendment 1 used 0.003 to 0.1 and chose 0.01, whose gain is within 5% of Adam's.

## Amendment 1: does stable rank or damage predict which trained decomposition passes?

![check against stable rank and damage](../experiments/broad_fragility/figures/amend1_rank.png)

**Reading it.** Neither does. In the left panel, SGD's stable rank of 3.5 sits near the LoRA's 1.9, far from Adam's
28.5, but SGD's trained decomposition fails with Adam's. So being less diffuse, in this sense, does not separate pass
from fail. In the right panel, SGD's trained decomposition does the least damage on plain text, 0.090 against the
LoRA's 0.124, and still fails. Its largest weight change is also the smallest of the three. Those were the LoRA's two
confounds in the first run, and SGD removes both. What sets the LoRA apart is untested. Candidates, named and not
asserted: its hard cap of rank 8 in every matrix; which matrices it moves; the total size of its trained move; and
how much of its update lies outside the subcomponents. SGD's fail rests mostly on the second block of 128 Pile rows,
which reads 2.93, while its first block reads 1.16.

In [1]:
# The reads behind the figures, from the two results files (no model).
import json
from pathlib import Path

res = Path("../experiments/broad_fragility/results")
r = json.loads((res / "broad_fragility.json").read_text())
a1 = json.loads((res / "amend1.json").read_text())
share = r["score"]["share_removed"]
reads = r["reads"] | {k: v for k, v in a1["reads"].items() if k.endswith("_sgd")}
print(f"{'state':16} {'check':>6} {'kept':>6} {'damage':>7} {'share removed':>14}")
for k, v in reads.items():
    s = f"{share[k.removeprefix('revert_')]:14.2f}" if k.startswith("revert_") else ""
    print(f"{k:16} {v['check']:6.2f} {v['s']:6.2f} {v['damage']:7.3f} {s}")
sh = r["shape"] | {"sgd": a1["shape"]["sgd"]}
print()
for arm in ("adam", "sgd", "lora"):
    print(f"{arm:5} gain {sh[arm]['gain']:.3f} stable rank {sh[arm]['stable_rank']['mean']:5.1f} "
          f"outside share {sh[arm]['outside_share']:.3f} (random {sh[arm]['outside_null']:.3f})")

state             check   kept  damage  share removed
adjoint_lora       0.95   0.67   0.024 
move_s0_x0.5       0.99  -0.01   0.003 
move_s0_x1         1.03  -0.06   0.009 
move_s1_x1         1.07  -0.07   0.010 
nothing_lora       2.65   1.00   0.121 
revert_all         1.00   0.00   0.000           1.00
revert_attn        2.70   1.01   0.133          -0.42
revert_common      1.04   0.76   0.021           0.96
revert_layer0      1.35   0.86   0.168           0.70
revert_layer1      1.75   1.02   0.175           0.38
revert_layer2      1.90   1.01   0.128           0.25
revert_layer3      2.07   0.82   0.098           0.11
revert_mlp         1.00   0.47   0.025           1.00
revert_random      1.03   1.00   0.054           0.97
revert_rare        1.45   1.04   0.111           0.63
trained            2.20   1.00   0.234 
tt_lora            1.27   0.96   0.124 
adjoint_sgd        0.99   0.73   0.026 
nothing_sgd        2.62   1.00   0.086 
tt_sgd             2.07   0.96   0.090 

adam 

## What we learned

The user's question has a plain answer on this task: being less diffuse is not enough. A rank-8 LoRA of the lowercase
task gives a trained decomposition that passes. An SGD fine-tune with almost as low a stable rank gives one that fails
like Adam's, while doing less damage on plain text. On the Adam update, the excess needs most of the MLP changes
together, partly offset by the attention changes. Ranked by importance at the unchanged model, the rarely important
subcomponents carry less of it than a random half. An isotropic random move of the same weight size passes, but it
changes the outputs far less, so brittleness along the directions training uses is not ruled out. For `GOALS.md`,
teacher training does not give a faithful decomposition of a broad full-matrix update, Adam or SGD. The one broad
update it handled is the LoRA's, for a reason not yet known.

## How compelling is the evidence

- **One adversary from one random start**, 20 steps: a lower bound. Every pass here could read higher under a longer
  search.
- **Noise floor.** The unchanged model's spread over three adversary seeds is 0.0748, a band from 1 to 1.075. E37's
  two batch orders differed by 14% in the check. Shares of excess removed less than 0.2 apart are not separated.
- **The random moves are matched in weight size only.** They do a twenty-fifth of the damage.
- **Reverts are not additive.** A share says what removing one group does with the rest in place.
- **Importance was read at the unchanged model on the check's rows**, not at the trained state or on other text.
- **Stable rank is an average** and says nothing about the actual rank or where the update acts.
- **One task, one batch order, one seed and one rate per SGD and LoRA fine-tune.** SGD's fail rests mostly on one
  block of 128 rows.

## Surprises

- **A random move of the same size passes.** It was predicted to fail at 1.8 to 3, and read 1.03 and 1.07. Untested
  causes: the check may track output change, which random directions barely produce. Or the adversary may exploit the
  particular directions teacher training chose.
- **The MLP changes alone carry all the excess**, an unpredicted result. Untested causes: the attention changes may
  compensate for the MLP changes on Pile text. Or the check may fall with the remaining damage.
- **The rarely important half removes less of the excess than a random half**, 0.63 against 0.97. Untested causes:
  that half may carry less of the weight change. The ranking may be wrong. Or any half revert may break the combination
  the excess needs.
- **The LoRA's trained decomposition passes, while SGD's fails.** Untested candidates: the hard rank cap, which
  matrices move, the total trained move, the outside share, or chance at one seed.
- **Layer 0's revert removed 0.70 of the excess**, a narrow miss of the 0.6 cap, inside the noise of a share.
- **Putting back a random half keeps the behaviour by mean log-probability and passes.** Its KL to the fine-tuned
  model is six times the trained decomposition's, so it is not a fix as it stands.

## What next

No run is proposed: the user decided against further training. If the LoRA question is taken up, these would
separate the candidates.

1. **SGD restricted to rank 8 per matrix.** A pass points to the hard rank cap; a fail points to the LoRA's optimiser
   or factorisation.
2. **A random move matched in output change.** If it fails, the check punishes any change of this functional size.
   If it passes, the fragility is specific to where teacher training puts the change.
3. **The LoRA and SGD arms at a second seed, and the check on a second set of Pile rows.**

## Scorecard

Filed in `results/predictions.json` and `results/predictions_amend1.json`, each before its run, unedited. SGD rows
from `amend1.json`, LoRA rows from `broad_fragility.json`. A prediction over both arms holds only if both halves hold.

| id | claim | read | verdict |
|---|---|---|---|
| P1 | a random move of the same size fails, mean of 2 seeds ≥ 1.61 | 1.049 | **miss**: passes, inside the band |
| P2 | the half-size move reads below the full-size mean | 0.988 against 1.049 | holds, within the noise |
| P3 | no single layer's revert removes more than 0.6 | layer 0: 0.705 | **miss**, narrow |
| P4 | rarely important half removes ≥ 0.2 more than a random half | 0.629 − 0.973 = −0.344 | **miss**, opposite way |
| P5 | SGD and LoRA stable rank below Adam's 28.5 | SGD 3.50; LoRA 1.93 | holds |
| P6 | SGD and LoRA trained decompositions fail, ≥ 1.61 | SGD 2.067; LoRA 1.273 | **miss**: LoRA half misses |
| P7 | SGD and LoRA do-nothing fail, ≥ 1.61 | SGD 2.623; LoRA 2.653 | holds |
| P8 | the chosen SGD rate's gain within 30% of Adam's | ratio 1.052 | holds |

## Not run

- A random move matched in damage or in output change; random moves along directions the inputs use.
- Reverts of single matrices, of each layer's MLP alone, or of importance halves within the MLP alone.
- Importance read at the trained state, or on lowercase text.
- A second adversary seed, a longer adversary, or other Pile rows on any state.
- A second seed or rate for the SGD or LoRA arm; SGD restricted to rank 8; a LoRA at another rank.
- The actual rank of each update; reverts or random moves on the SGD or LoRA trained decompositions.
- The LoRA's teacher-trained state is not on disk whole: both copies are truncated, so CPU follow-ups need a rerun.

**Provenance.** Two RunPod H100 80GB HBM3 jobs: the first run's log from 15:44 to 16:08, Amendment 1's from 16:38 to
16:50. After the first, the plugin lost SSH to the pod; the user copied the results file and the log by hand. Results
`results/broad_fragility.json` and `results/amend1.json`, both status done; logs `results/large/run.log` and
`results/large/amend1.log`.